In [1]:
# import gi
# import os
import numpy as np
from tqdm import tqdm
import librosa
import pandas as pd
# from aquatk.metrics.PEAQ.peaq import PEAQ
# from aquatk.metrics.PEAQ.peaq_basic import process_audio_files
import torch
# os.environ["GST_PLUGIN_PATH"] = "/usr/local/lib/gstreamer-1.0"

# gi.require_version('Gst', '1.0')

# from gi.repository import Gst

# Gst.init(None)

from torchmetrics.audio import SignalNoiseRatio
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio

In [2]:
import gi
import os

os.environ["GST_PLUGIN_PATH"] = "/usr/local/lib/gstreamer-1.0"

gi.require_version('Gst', '1.0')
from gi.repository import Gst

Gst.init(None)

registry = Gst.Registry.get()
feature = registry.lookup_feature("peaq")

In [3]:
def peaq_score(ref, test):
    pipeline_str = (
        f"filesrc location=\"{os.path.abspath(ref)}\" ! decodebin ! audioconvert ! audioresample ! "
        f"audio/x-raw,format=F32LE,rate=48000,channels=1 ! queue ! peaq name=p "
        f"filesrc location=\"{os.path.abspath(test)}\" ! decodebin ! audioconvert ! audioresample ! "
        f"audio/x-raw,format=F32LE,rate=48000,channels=1 ! queue ! p.test"
    )
    pipeline = Gst.parse_launch(pipeline_str)
    peaq = pipeline.get_by_name("p")
    bus = pipeline.get_bus()
    pipeline.set_state(Gst.State.PLAYING)
    bus.timed_pop_filtered(Gst.CLOCK_TIME_NONE, Gst.MessageType.EOS | Gst.MessageType.ERROR)
    odg = peaq.get_property("odg")
    pipeline.set_state(Gst.State.NULL)
    return odg

In [4]:
# peaq = PEAQ()

In [5]:
si_sdr = ScaleInvariantSignalDistortionRatio()
snr = SignalNoiseRatio()

In [6]:
def lsd(ref, test):
    s1 = librosa.stft(ref, n_fft=2048, hop_length=512)
    s2 = librosa.stft(test, n_fft=2048, hop_length=512)
    
    p1 = np.abs(s1)**2
    p2 = np.abs(s2)**2
    
    log_diff = 10 * np.log10(p1 + 1e-12) - 10 * np.log10(p2 + 1e-12)
    
    return np.mean(np.sqrt(np.mean(log_diff**2, axis=0)))

In [7]:
sr = 48000
base_path = "data"
num_samples = 32

results = []

for i in tqdm(range(1, num_samples+1)):
    file_name = f"{i}.wav"

    clean_path = os.path.join(base_path, "clean", file_name)
    noisy_path = os.path.join(base_path, "noisy", file_name)
    denoised_path = os.path.join(base_path, "denoised", file_name)
    specsub_path = os.path.join(base_path, "spectral_sub", file_name)
    
    clean, _ = librosa.load(clean_path, sr=sr, res_type='soxr_hq')
    noisy, _ = librosa.load(noisy_path, sr=sr, res_type='soxr_hq')
    pred, _ = librosa.load(denoised_path, sr=sr, res_type='soxr_hq')
    specsub, _ = librosa.load(specsub_path, sr=sr, res_type='soxr_hq')

    min_len = min(len(clean), len(noisy), len(pred), len(specsub))
    clean, noisy, pred, specsub = clean[:min_len], noisy[:min_len], pred[:min_len], specsub[:min_len]
    
    result = {
        "id": i,
        
        "Noisy_PEAQ": peaq_score(clean_path, noisy_path),
        # "Noisy_PEAQ": peaq.analyze_files(clean_path, noisy_path).odg,
        # "Noisy_PEAQ_basic": process_audio_files(clean_path, noisy_path)["ODG_list"][-1],
        "Noisy_SI_SDR": si_sdr(torch.from_numpy(clean), torch.from_numpy(noisy)).item(),
        "Noisy_SNR": snr(torch.from_numpy(clean), torch.from_numpy(noisy)).item(),
        "Noisy_LSD": lsd(clean, noisy),
        
        "Model_PEAQ": peaq_score(clean_path, denoised_path),
        # "Model_PEAQ": peaq.analyze_files(clean_path, denoised_path).odg,
        # "Model_PEAQ_basic": process_audio_files(clean_path, denoised_path)["ODG_list"][-1],
        "Model_SI_SDR": si_sdr(torch.from_numpy(clean), torch.from_numpy(pred)).item(),
        "Model_SNR": snr(torch.from_numpy(clean), torch.from_numpy(pred)).item(),
        "Model_LSD": lsd(clean, pred),
        
        "SpecSub_PEAQ": peaq_score(clean_path, specsub_path),
        # "SpecSub_PEAQ": peaq.analyze_files(clean_path, specsub_path).odg,
        # "SpecSub_PEAQ_basic": process_audio_files(clean_path, specsub_path)["ODG_list"][-1],
        "SpecSub_SI_SDR": si_sdr(torch.from_numpy(clean), torch.from_numpy(specsub)).item(),
        "SpecSub_SNR": snr(torch.from_numpy(clean), torch.from_numpy(specsub)).item(),
        "SpecSub_LSD": lsd(clean, specsub),    
    }
    results.append(result)


  0%|          | 0/32 [00:00<?, ?it/s]

   BandwidthRefB: 871.809917
  BandwidthTestB: 871.796143
      Total NMRB: 9.913338
    WinModDiff1B: 14.699967
            ADBB: 2.287991
            EHSB: 1.157050
    AvgModDiff1B: 13.629245
    AvgModDiff2B: 337.842262
   RmsNoiseLoudB: 0.262428
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.282
   BandwidthRefB: 871.809917
  BandwidthTestB: 871.796143
      Total NMRB: 9.908873
    WinModDiff1B: 14.701867
            ADBB: 2.287795
            EHSB: 1.155545
    AvgModDiff1B: 13.639860
    AvgModDiff2B: 337.408222
   RmsNoiseLoudB: 0.262247
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.282
   BandwidthRefB: 908.025992
  BandwidthTestB: 906.357045
      Total NMRB: -2.080573
    WinModDiff1B: 11.634937
            ADBB: 1.758312
            EHSB: 6.408337
    AvgModDiff1B: 9.835820
    AvgModDiff2B: 76.127023
   RmsNoiseLoudB: 0.272179
           MFPDB: 1.000000
  RelDistFramesB: 0.980848
Objective Differ

  3%|▎         | 1/32 [00:01<00:53,  1.73s/it]

   BandwidthRefB: 895.110807
  BandwidthTestB: 894.993160
      Total NMRB: 4.500260
    WinModDiff1B: 42.947172
            ADBB: 2.777972
            EHSB: 5.752013
    AvgModDiff1B: 44.132534
    AvgModDiff2B: 335.738194
   RmsNoiseLoudB: 1.330110
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.901
   BandwidthRefB: 895.146175
  BandwidthTestB: 895.023224
      Total NMRB: 4.497006
    WinModDiff1B: 42.948344
            ADBB: 2.777964
            EHSB: 5.750880
    AvgModDiff1B: 44.167327
    AvgModDiff2B: 335.489947
   RmsNoiseLoudB: 1.332723
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.901
   BandwidthRefB: 868.766402
  BandwidthTestB: 868.759913
      Total NMRB: 10.055210
    WinModDiff1B: 14.059502
            ADBB: 2.274513
            EHSB: 1.096253
    AvgModDiff1B: 12.890368
    AvgModDiff2B: 299.127036
   RmsNoiseLoudB: 0.267370
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Diff

  6%|▋         | 2/32 [00:02<00:35,  1.19s/it]

...

   BandwidthRefB: 896.056222
  BandwidthTestB: 895.930285
      Total NMRB: -0.082679
    WinModDiff1B: 38.527572
            ADBB: 2.687942
            EHSB: 2.062496
    AvgModDiff1B: 38.591405
    AvgModDiff2B: 91.385571
   RmsNoiseLoudB: 1.678979
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.699
   BandwidthRefB: 896.047940
  BandwidthTestB: 895.922097
      Total NMRB: -0.081738
    WinModDiff1B: 38.524830
            ADBB: 2.687935
            EHSB: 2.063361
    AvgModDiff1B: 38.589491
    AvgModDiff2B: 91.387222
   RmsNoiseLoudB: 1.678930
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.699
   BandwidthRefB: 867.807942
  BandwidthTestB: 867.798556
      Total NMRB: -15.268686
    WinModDiff1B: 1.023223
            ADBB: 0.868656
            EHSB: 0.199067
    AvgModDiff1B: 0.704844
    AvgModDiff2B: 1.404020
   RmsNoiseLoudB: 0.042838
           MFPDB: 0.990513
  RelDistFramesB: 0.080144
Objective Differe

 97%|█████████▋| 31/32 [00:20<00:00,  1.53it/s]

   BandwidthRefB: 871.187970
  BandwidthTestB: 871.175188
      Total NMRB: 2.434512
    WinModDiff1B: 22.399713
            ADBB: 1.929535
            EHSB: 0.152256
    AvgModDiff1B: 1.464826
    AvgModDiff2B: 20.776288
   RmsNoiseLoudB: 0.720653
           MFPDB: 0.999984
  RelDistFramesB: 0.169173
Objective Difference Grade: -0.534
   BandwidthRefB: 871.048084
  BandwidthTestB: 871.035312
      Total NMRB: 2.431258
    WinModDiff1B: 22.391137
            ADBB: 1.929535
            EHSB: 0.152283
    AvgModDiff1B: 1.464608
    AvgModDiff2B: 20.758225
   RmsNoiseLoudB: 0.720377
           MFPDB: 0.999984
  RelDistFramesB: 0.169046
Objective Difference Grade: -0.535
   BandwidthRefB: 907.193233
  BandwidthTestB: 905.538346
      Total NMRB: -14.425336
    WinModDiff1B: 4.193263
            ADBB: 1.257016
            EHSB: 0.247744
    AvgModDiff1B: 1.801475
    AvgModDiff2B: 3.632140
   RmsNoiseLoudB: 0.146099
           MFPDB: 0.998322
  RelDistFramesB: 0.091729
Objective Difference 

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

   BandwidthRefB: 891.209774
  BandwidthTestB: 891.099248
      Total NMRB: 0.232169
    WinModDiff1B: 55.852992
            ADBB: 2.796113
            EHSB: 0.390821
    AvgModDiff1B: 56.943912
    AvgModDiff2B: 132.871393
   RmsNoiseLoudB: 2.171889
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.088
   BandwidthRefB: 891.214125
  BandwidthTestB: 891.103681
      Total NMRB: 0.230841
    WinModDiff1B: 55.847686
            ADBB: 2.796093
            EHSB: 0.390610
    AvgModDiff1B: 56.944692
    AvgModDiff2B: 132.909056
   RmsNoiseLoudB: 2.171300
           MFPDB: 1.000000
  RelDistFramesB: 1.000000
Objective Difference Grade: -3.088


In [8]:
df = pd.DataFrame(results)

metrics = ["PEAQ", 
        #    "PEAQ", 
           # "PEAQ_basic", 
           "SI_SDR", "SNR", "LSD"]
audios = ["Noisy", "Model", "SpecSub"]

final_results = []

for m in metrics:
    row = {"Metric": m}
    for audio in audios:
        col = f"{audio}_{m}"
        mean = df[col].mean()
        std = df[col].std()
        row[audio] = f"{mean:.3f} (+-{std:.3f})"
    final_results.append(row)

In [9]:
print(pd.DataFrame(final_results))

   Metric             Noisy             Model            SpecSub
0    PEAQ  -1.842 (+-1.084)  -1.408 (+-0.977)   -3.593 (+-0.376)
1  SI_SDR  29.582 (+-6.969)  21.309 (+-2.767)  -24.833 (+-7.174)
2     SNR  29.508 (+-7.024)  21.314 (+-2.784)   -6.770 (+-0.979)
3     LSD  15.829 (+-3.898)   7.957 (+-1.644)   14.256 (+-2.046)


In [10]:
df.to_csv("raw_metrics.csv", index=False)
pd.DataFrame(final_results).to_csv("summary.csv", index=False)